# Objectives

- View the Liquidity provider token as derivative
- Compare that to a standard derivative 
- Understand why an LP might provide liquidity other than to simply earn fees

# Prerequisites

- Metamask
- Web3 python

# Derivatives and their payoff functions

Investing has grown more complicated in recent decades with the creation of numerous derivative instruments offering new ways to manage money. The use of derivatives to hedge risk or improve returns has been around for generations.

**Options** are the simplest derivative investment. Their value is tied to the value of the contract's underlying security. Options give a buyer the opportunity to buy or sell the underlying security. The investor does not own the underlying asset but they make a bet on the direction of its price movement.

The payoff function of an option shows the profit/loss obtained from an option depending on its market price.

There are many types of derivative instruments, including options, swaps, futures, and forward contracts. Derivatives have numerous uses and various levels of risks but are generally considered a sound way to participate in the financial markets.

## Example: Call/Put Options

Here, we only discuss European options.

### Call Option 

A call option is a type of option that gives the holder the right to **buy** the underlying asset at a specified price, known as the strike price, before the option expires, with the cost of the premium. If the price of the underlying asset rises above the strike price, the holder can exercise the option and purchase the asset at the lower strike price, then sell it at the higher market price for a profit. If the price does not rise above the strike price, the holder can choose not to exercise the option and simply let it expire worthless. 

![Call Option Payoff Function](./img/calloption.png)

- $S$ is the price of the underlying asset at expiration
- $X$ is the strike price of the option
- Breakeven point: $S = X + \text{option price}$

### Put Option

On the other hand, a put option is a type of option that gives the holder the right to **sell** the underlying asset at a specified price, before the option expires. If the price of the underlying asset falls below the strike price, the holder can exercise the option and sell the asset at the higher strike price, then buy it back at the lower market price for a profit. If the price does not fall below the strike price, the holder can choose not to exercise the option and simply let it expire worthless. In both cases, the holder pays a premium for the option contract, which is the price of the option, and if the option is not exercised, this premium represents a loss for the holder.

![Put Option Payoff Function](./img/putoption.png)


- Put token USTUSD as 0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c
- Put token MiniDAI as 0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21
- Put pool address as 0xc9D9C8dD62211D2d4F5E941Fa9A91AF686B290c4

## Connect to web3 API

In [ ]:
from web3 import Web3
import json
import os

infura_key = ''
wallet_public_address = Web3.to_checksum_address('')
wallet_private_key = ''

USTUSD_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))
print("Connected to Sepolia Testnet:", web3)

abi_file_path = os.path.join('./abis.json')
try:
    with open(abi_file_path, 'r', encoding='utf-8') as f:
        abi_data = json.load(f)
    print("ABI loaded successfully.")
except Exception as e:
    print(f"Error loading ABI: {e}")


## Identify the MiniDAI/USTUSD pool

In [ ]:
factory_addr = Web3.to_checksum_address('0x0227628f3F023bb0B980b67D528571c95c6DaC1c') #Uniswap V3 factory contract address sepolia
abi_factory = abi_data["abi_factory"]
factory_contract = web3.eth.contract(factory_addr, abi=abi_factory)

dai_token_address = web3.to_checksum_address('0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21')
USTUSD_token_address = web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
abi_dai = abi_data["abi_dai"]
abi_USTUSD = abi_data["abi_USTUSD"]
MiniDAI_contract = web3.eth.contract(dai_token_address, abi=abi_dai)
USTUSD_contract = web3.eth.contract(USTUSD_token_address, abi=abi_USTUSD)

def get_pool_address(tokenA, tokenB, fee,factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, fee).call()
    return pool_address

MiniDAI_USTUSD_pool_address = get_pool_address(USTUSD_token_address, dai_token_address, 100, factory_contract)

print("Pool with fee 0.01% :",MiniDAI_USTUSD_pool_address)


In [ ]:
abi_pool = abi_data["abi_pool"]

MiniDAI_USTUSD_pool_V3 = web3.eth.contract(address=MiniDAI_USTUSD_pool_address, abi=abi_pool)

sqrtPriceX96 = MiniDAI_USTUSD_pool_V3.functions.slot0().call()[0]
raw_price = (sqrtPriceX96 ** 2) / (2 ** 192)

print(f"1 MiniDAI = {raw_price} USTUSD")


## Compare Uniswap V2 and V3

Uniswap V2 and V3 has differernt liquidity function, which is generated by the range of pool. 

- In Uniswap V2, we set the range is $\left( 0,\infty \right)$
- In Uniswap V3, we set the price range $\left[ p_a,p_b\right]$ at the beginning, which is can be seen in the following picture. 

The following picture depicts the relationship for a position on a range $\left[ p_a,p_b\right]$ and a current price $p_c \in \left[ p_a,p_b\right]$. $x_{real}$ and $y_{real}$
denote the position’s real reserves. If you are interested in more details, please see the whitepaper [here](https://app.uniswap.org/whitepaper-v3.pdf).

![Uniswap V3 Liquidity Function](./img/UniswapV3.png)

When calculate the payoff with different price, we first need the value of price, there are two ways to get. Traditionally, we calculate the price of Uniswap V2 and V3 by definition. 

In Uniswap V2, liquidity was distributed uniformly along the $x \cdot y=K$, then we know that $(x-\Delta x)(y+\Delta y) = x \cdot y$.

Then it's easy to conduct the price in Uniswap V2: $$\frac{\Delta y}{\Delta x} = \frac{y}{x}. $$ 

In Uniswap V3, The amount of liquidity provided can be measured by the value $L$, which is equal to $\sqrt{K}$. The real reserves of a position are described by the curve: 

$ (x+\frac{L}{\sqrt{p_b}})(y + L \sqrt{p_a})=L^2 .$ 

Similarly, we can calculate the price in Uniswap V3: 

$ \frac{\Delta y}{\Delta x} = \frac{y+L\sqrt{p_a}}{x+\frac{L}{\sqrt{p_b}}} $. 

However, because of the existence of ``Unclaimed fees`` in the Uniswap pool, this traditional way is not credible.

So, we have to use another way - ``slot0`` function. Please note the positions of tokenA and tokeB in this method. 



**Payoff calculation in unit of USTUSD**

$$p = \frac{y}{x}$$
$$ \text{Payoff} = y + p \times x $$

For V2, we can calculate the payoff by using the price $p$ and the constant $k$:

$$ \text{Payoff} = \sqrt{k \times p} + p \times \sqrt{\frac{k}{p}} $$
$$ \text{Payoff} = 2\sqrt{k \times p} $$

## Define the payoff function

In [ ]:
import math

x0 = MiniDAI_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
y0 = USTUSD_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
k = x0 * y0

def getLPpayoff(mode="v3"):
    '''
    p = price of MiniDAI in USTUSD
    payoff = the value of LP token in USTUSD
    '''
    x = MiniDAI_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
    y = USTUSD_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
    p = MiniDAI_USTUSD_pool_V3.functions.slot0().call()[0]**2 / 2**192

    if mode.lower() == "v3": # todo: need to calculate the payoff of V3

    elif mode.lower() == "v2": # todo: need to calculate the payoff of V2

    else:
        raise ValueError("mode must be v2 or v3")

    return {'price': p, 'payoff': payoff}

## Calculate LP token payoffs

We now calculate LP payoffs for a range of prices. To do that, the TA will manipualte the pool price by doing large swaps.

**Current Pool V3**
- 500000 MiniDAI
- Price: 1 MiniDAI $\approx$ 1 USTUSD
- min price: 0.8
- max price: 1.4

In [ ]:
import json
'''
{
v2: [
    {
        "price": 1.0,
        "payoff": 1000.0
    },
    {
        "price": 1.1,
        "payoff": 900.0
    },
    ...
],
v3: [
    {
        "price": 1.0,
        "payoff": 1000.0
    },
    {
        "price": 1.1,
        "payoff": 900.0
    },
    ...
]
}
'''
def append_to_json(data, v2_or_v3, filename='payoff_data.json'):
    if not os.path.exists(filename):
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump({"v2": [], "v3": []}, f, ensure_ascii=False, indent=4)
    with open(filename, 'r', encoding='utf-8') as f:
        existing_data = json.load(f)
    existing_data[v2_or_v3].append(data)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(existing_data, f, ensure_ascii=False, indent=4)
def load_data_from_json(filename='payoff_data.json'):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        data = {"v2": [], "v3": []}
    return data

### Waiting for the TA to manipulate the price...
TA will manipulate the price by doing large swaps for several times. After each swap, we will calculate the payoff of the LP token and record the price and payoff in a json file.

In [ ]:
data = load_data_from_json()
iteration = len(data['v2']) + 1
print(f"Current iteration: {iteration}")

result_v2 = getLPpayoff(mode="V2")
print(f"V2 result: {result_v2}")
append_to_json(result_v2, v2_or_v3="v2")

result_v3 = getLPpayoff(mode="V3")
print(f"V3 result: {result_v3}")
append_to_json(result_v3, v2_or_v3="v3")

### Waiting for the TA to manipulate the price...

## Plot LP token payoffs

In [ ]:
import matplotlib.pyplot as plt

data = load_data_from_json('payoff_data.json')

v2_prices = sorted([item['price'] for item in data['v2']])
v2_payoffs = sorted([item['payoff'] for item in data['v2']])

v3_prices = sorted([item['price'] for item in data['v3']])
v3_payoffs = sorted([item['payoff'] for item in data['v3']])

# draw the plot for V2
plt.figure(figsize=(10, 6))
plt.plot(v2_prices, v2_payoffs, marker='o', color='blue', label='V2 LP Payoff')
plt.xlabel('MiniDAI price in USTUSD')
plt.ylabel('LP payoff in USTUSD')
plt.title('Calculate LP token payoff of V2')
plt.legend()
plt.grid(True)
plt.show()

# draw the plot for V3
plt.figure(figsize=(10, 6))
plt.plot(v3_prices, v3_payoffs, marker='s', color='orange', label='V3 LP Payoff')
plt.xlabel('MiniDAI price in USTUSD')
plt.ylabel('LP payoff in USTUSD')
plt.title('Calculate LP token payoff of V3')
plt.legend()
plt.grid(True)
plt.show()

Make sure to save the plot obtained above on your machine.

#### Both payoff curves are fundamentally concave functions:


**Uniswap V2:** Liquidity is distributed uniformly across the entire price range from $0$ to infinity ($\infty$). As a result, its local curvature is extremely low, making the curve approximate a straight line within a narrow price window.

**Uniswap V3 (Concentrated Liquidity = A Leveraged Curve):** V3 compresses all the capital—which would otherwise be dispersed to infinity—strictly into the localized interval of $[0.8, 1.4]$. Consequently, its curvature (representing the option's Gamma) is significantly amplified."


## Replicating market makers

Turns out you can generate a rich family of payoffs using a CFMM. In fact there is a one-to-one mapping between concave payoffs and CFMM bonding curves! Convex payoffs can be obtained by shorting an LP token.

This implies that LPs might not invest money in pools to just earn fees, they might also want to take bets on prices of the underlying tokens, or hedge their other investments in some way. We can tailor the bonding curve based on the sort of payoff function the LP is looking for.

If anyone is interested to know this mapping and how it is obtained, see [this paper](https://arxiv.org/pdf/2103.14769.pdf). 



## Perpetual

**1. What is a Perpetual?**
A perpetual is a **synthetic asset** designed to track the price of an underlying asset. It acts as an agreement between two parties to buy or sell an asset at a future date, based on the price at the time the position was opened. 
* **Long Position:** Betting the price will go up.
* **Short Position:** Betting the price will go down.
* **No Real Asset Exchange:** You are trading a contract that mimics the asset's price movements, without ever needing to hold the actual underlying asset.

**2. Collateral, Leverage, and Liquidation**
Because no actual assets change hands until settlement, you do not need to put up the full value of the trade.
* **Collateral:** You only need to deposit a fraction of the position's value as collateral. This allows for high **leverage**.
* **Liquidation:** If the market moves against you and your losses exceed a certain threshold, the protocol will **liquidate** your position. It seizes your collateral to pay the winning side and sells your position to someone providing fresh collateral.

**3. How Profits and Losses are Realized**
You realize your profit or loss simply by closing your position. 
* The payout corresponds exactly to the profit or loss you would have made if you bought/sold the underlying asset directly at your leveraged size. 
* Depending on the specific protocol's design, "closing" your position means selling it back to an **Order Book**, an **AMM (Automated Market Maker)**, or settling against a **Liquidity Pool**.

**4. The Funding Rate: How the Price Stays Pegged**
* **The Problem:** If market sentiment is extremely bullish, high demand might push the price of the "Long" perpetual contract significantly higher than the actual underlying asset's price. 
* **The Solution:** Protocols use a **Funding Rate mechanism**. It charges periodic fees to the side causing the imbalance (the overvalued positions) and pays those fees to the opposing side (the undervalued positions).
* **The Effect:** This creates a financial incentive that counteracts extreme market sentiment, keeping the demand for long and short positions balanced and ensuring the perpetual's price stays pegged to the underlying asset.

**5. Perpetuals vs. Traditional Futures**
While both are considered futures contracts, they have one major difference:
* **No Expiry Date:** Traditional futures expire on a specific date, at which point assets must be exchanged. Perpetuals have **no expiry**. You can hold your position forever, provided you have enough collateral to avoid liquidation.